# Baseline RoCoLe HSV masked: mayoría, logística y Random Forest

Este cuaderno arma un baseline honesto con máscara COCO sumando las siete medidas RGB/GLCM (sin VARI) y dieciséis bins del histograma HSV. Comparamos mayoría, logística y bosque. Hiperparámetros solo en val; test una sola vez. Prefijo de resultados: rocole_baseline_hsv_masked_.


Fijamos la semilla del proyecto para que el azar del bosque se pueda repetir.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import StandardScaler


In [ ]:
# Semilla fija del proyecto
seed = 42

seed


Localizamos las dos tablas featured enmascaradas (RGB/GLCM y HSV) y la carpeta de resultados, relativas al repo.


In [ ]:
rgb_glcm_path = Path("../../data/featured/rocole/rgb_glcm_features_masked.csv")
hsv_hist_path = Path("../../data/featured/rocole/hsv_hist_features_masked.csv")
results_dir = Path("../../experiments/results")
results_dir.mkdir(parents=True, exist_ok=True)

print("rgb_glcm_path:", rgb_glcm_path.resolve(), "existe:", rgb_glcm_path.exists())
print("hsv_hist_path:", hsv_hist_path.resolve(), "existe:", hsv_hist_path.exists())
print("results_dir:", results_dir.resolve())
rgb_glcm_path, hsv_hist_path, results_dir


Cargamos la mesa RGB/GLCM enmascarada. De aquí saldrán el split, la etiqueta binaria y el grupo de enlace.


In [ ]:
rgb_glcm_features = pd.read_csv(rgb_glcm_path)

print("filas rgb/glcm:", len(rgb_glcm_features))
rgb_glcm_features.head()


## Cargar tabla HSV

Cargamos la mesa HSV enmascarada con los dieciséis bins del histograma.


In [ ]:
# hsv_hist_path

hsv_hist_features = pd.read_csv(hsv_hist_path)

print("filas hsv:", len(hsv_hist_features))
hsv_hist_features.head()


Dejamos fuera las filas con error en cualquiera de las dos mesas y unimos por File. Conservamos split, etiqueta binaria y grupo de enlace del RGB/GLCM.


In [ ]:
# rgb_glcm_features, hsv_hist_features

rgb_ok = rgb_glcm_features.copy()
if "error" in rgb_ok.columns:
    rgb_error_text = rgb_ok["error"].astype("string").fillna("").str.strip()
    rgb_ok = rgb_ok.loc[rgb_error_text == ""].copy()

hsv_ok = hsv_hist_features.copy()
if "error" in hsv_ok.columns:
    hsv_error_text = hsv_ok["error"].astype("string").fillna("").str.strip()
    hsv_ok = hsv_ok.loc[hsv_error_text == ""].copy()

hsv_feature_names = [c for c in hsv_ok.columns if c.startswith("hsv_")]
hsv_keep = hsv_ok[["File"] + hsv_feature_names].copy()

meta_columns = ["File", "split", "Binary.Label", "link_group"]
rgb_measure_columns = [
    "exg_mean",
    "exg_std",
    "ngrdi_mean",
    "ngrdi_std",
    "glcm_contrast",
    "glcm_homogeneity",
    "glcm_energy",
]
rgb_keep = rgb_ok[meta_columns + rgb_measure_columns].copy()

leaf_features = rgb_keep.merge(hsv_keep, on="File", how="inner")
print("filas rgb sin error:", len(rgb_ok))
print("filas hsv sin error:", len(hsv_ok))
print("filas tras merge inner por File:", len(leaf_features))

leaf_features.head()


Miramos de un vistazo filas y columnas para confirmar que llegó el merge con las veintitrés medidas.


In [ ]:
# leaf_features

print("filas:", len(leaf_features))
print("columnas:", list(leaf_features.columns))

leaf_features.info()


Contamos cuántas hojas hay por etiqueta binaria en toda la mesa unida (sana frente a afectada).


In [ ]:
# leaf_features

label_counts = leaf_features["Binary.Label"].value_counts()

label_counts


Contamos cuántas hojas caen en train, val y test según el split ya guardado.


In [ ]:
# leaf_features

split_counts = leaf_features["split"].value_counts()

split_counts


Listamos las veintitrés medidas: siete RGB/GLCM sin VARI más dieciséis bins HSV (ocho de matiz, cuatro de saturación y cuatro de brillo).


In [ ]:
feature_columns = [
    "exg_mean",
    "exg_std",
    "ngrdi_mean",
    "ngrdi_std",
    "glcm_contrast",
    "glcm_homogeneity",
    "glcm_energy",
    "hsv_h_b0",
    "hsv_h_b1",
    "hsv_h_b2",
    "hsv_h_b3",
    "hsv_h_b4",
    "hsv_h_b5",
    "hsv_h_b6",
    "hsv_h_b7",
    "hsv_s_b0",
    "hsv_s_b1",
    "hsv_s_b2",
    "hsv_s_b3",
    "hsv_v_b0",
    "hsv_v_b1",
    "hsv_v_b2",
    "hsv_v_b3",
]

print("n features:", len(feature_columns))
feature_columns


Apartamos las hojas de aprender con split igual a train.


In [ ]:
# leaf_features

train_leaves = leaf_features.loc[leaf_features["split"] == "train"].copy()

print("train:", len(train_leaves))
train_leaves.head()


Apartamos las hojas de validación con split igual a val.


In [ ]:
# leaf_features

val_leaves = leaf_features.loc[leaf_features["split"] == "val"].copy()

print("val:", len(val_leaves))
val_leaves.head()


Apartamos las hojas de prueba con split igual a test y las dejamos guardadas hasta el final.


In [ ]:
# leaf_features

test_leaves = leaf_features.loc[leaf_features["split"] == "test"].copy()

print("test:", len(test_leaves))
test_leaves.head()


Contamos sanas y afectadas solo en train, para ver el desbalance que aprenderá el modelo.


In [ ]:
# train_leaves

train_label_counts = train_leaves["Binary.Label"].value_counts()

train_label_counts


Contamos sanas y afectadas en val.


In [ ]:
# val_leaves

val_label_counts = val_leaves["Binary.Label"].value_counts()

val_label_counts


Contamos sanas y afectadas en test (solo inventario; aún no medimos acierto).


In [ ]:
# test_leaves

test_label_counts = test_leaves["Binary.Label"].value_counts()

test_label_counts


Sacamos de train la matriz de las veintitrés medidas, manteniendo el mismo orden de filas que las etiquetas.


In [ ]:
# train_leaves, feature_columns

train_features = train_leaves[feature_columns].copy()

print(train_features.shape)
train_features.head()


Sacamos de train la etiqueta binaria, en el mismo orden de filas.


In [ ]:
# train_leaves

train_labels = train_leaves["Binary.Label"].copy()

print(train_labels.shape)
train_labels.head()


Sacamos de val las veintitrés medidas.


In [ ]:
# val_leaves, feature_columns

val_features = val_leaves[feature_columns].copy()

print(val_features.shape)
val_features.head()


Sacamos de val la etiqueta binaria.


In [ ]:
# val_leaves

val_labels = val_leaves["Binary.Label"].copy()

print(val_labels.shape)
val_labels.head()


Sacamos de test las veintitrés medidas y las dejamos sin mirar acierto.


In [ ]:
# test_leaves, feature_columns

test_features = test_leaves[feature_columns].copy()

print(test_features.shape)
test_features.head()


Sacamos de test la etiqueta binaria.


In [ ]:
# test_leaves

test_labels = test_leaves["Binary.Label"].copy()

print(test_labels.shape)
test_labels.head()


Como piso tonto, un clasificador de mayoría: siempre predice la clase más frecuente de train.


In [ ]:
majority_model = DummyClassifier(strategy="most_frequent")
majority_model.fit(train_features, train_labels)

majority_model


Medimos ese piso en val (f1 macro y un reporte corto) para saber qué es no saber nada.


In [ ]:
majority_val_predictions = majority_model.predict(val_features)
majority_val_f1 = f1_score(val_labels, majority_val_predictions, average="macro")
majority_val_report = classification_report(val_labels, majority_val_predictions)
majority_val_metrics = {
    "f1_macro": majority_val_f1,
    "report": majority_val_report,
}

print("f1_macro val (mayoría):", majority_val_metrics["f1_macro"])
print(majority_val_metrics["report"])
majority_val_metrics


Ajustamos un escalador solo con las medidas de train (la logística es sensible a la escala).


In [ ]:
feature_scaler = StandardScaler()
feature_scaler.fit(train_features)

feature_scaler


Transformamos las medidas de train con ese escalador ya ajustado.


In [ ]:
# feature_scaler, train_features

train_features_scaled = feature_scaler.transform(train_features)

print(train_features_scaled.shape)
train_features_scaled


Transformamos las medidas de val con el mismo escalador de train.


In [ ]:
# feature_scaler, val_features

val_features_scaled = feature_scaler.transform(val_features)

print(val_features_scaled.shape)
val_features_scaled


Probamos en val unos pocos valores de C para la logística (0.01, 0.1, 1, 10, 100), con peso de clase balanceado.


In [ ]:
logistic_C_grid = [0.01, 0.1, 1, 10, 100]

logistic_val_scores = []
for C in logistic_C_grid:
    candidate = LogisticRegression(
        C=C,
        class_weight="balanced",
        max_iter=1000,
        random_state=seed,
    )
    candidate.fit(train_features_scaled, train_labels)
    val_pred = candidate.predict(val_features_scaled)
    score = f1_score(val_labels, val_pred, average="macro")
    logistic_val_scores.append({"C": C, "f1_macro": score})

logistic_val_scores


Nos quedamos con el C que mejor f1 macro dio en val.


In [ ]:
# logistic_val_scores

best_logistic_row = max(logistic_val_scores, key=lambda row: row["f1_macro"])
best_logistic_C = best_logistic_row["C"]

print("mejor C:", best_logistic_C, "f1_macro val:", best_logistic_row["f1_macro"])
best_logistic_C


Reajustamos la logística final con ese C, solo sobre train escalado.


In [ ]:
# train_features_scaled, train_labels, best_logistic_C

logistic_model = LogisticRegression(
    C=best_logistic_C,
    class_weight="balanced",
    max_iter=1000,
    random_state=seed,
)
logistic_model.fit(train_features_scaled, train_labels)

logistic_model


Guardamos las métricas de la logística en val (f1 macro y reporte).


In [ ]:
# logistic_model, val_features_scaled, val_labels

logistic_val_predictions = logistic_model.predict(val_features_scaled)
logistic_val_f1 = f1_score(val_labels, logistic_val_predictions, average="macro")
logistic_val_report = classification_report(val_labels, logistic_val_predictions)
logistic_val_metrics = {
    "f1_macro": logistic_val_f1,
    "report": logistic_val_report,
}

print("f1_macro val (logística):", logistic_val_metrics["f1_macro"])
print(logistic_val_metrics["report"])
logistic_val_metrics


Definimos una rejilla corta de árboles (100, 200) y profundidad (sin tope, 5, 10) para el bosque, con la semilla fija.


In [ ]:
# seed

rf_param_grid = [
    {"n_estimators": n_estimators, "max_depth": max_depth, "random_state": seed}
    for n_estimators in [100, 200]
    for max_depth in [None, 5, 10]
]

rf_param_grid


Probamos cada par de la rejilla en val midiendo f1 macro (bosque sobre medidas sin escalar; el RF no necesita el escalador), con peso de clase balanceado.


In [ ]:
rf_val_scores = []
for params in rf_param_grid:
    candidate = RandomForestClassifier(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        class_weight="balanced",
        random_state=params["random_state"],
    )
    candidate.fit(train_features, train_labels)
    val_pred = candidate.predict(val_features)
    score = f1_score(val_labels, val_pred, average="macro")
    rf_val_scores.append(
        {
            "n_estimators": params["n_estimators"],
            "max_depth": params["max_depth"],
            "f1_macro": score,
        }
    )

rf_val_scores


Elegimos el par de árboles y profundidad con mejor f1 macro en val.


In [ ]:
# rf_val_scores

best_rf_row = max(rf_val_scores, key=lambda row: row["f1_macro"])
best_rf_params = {
    "n_estimators": best_rf_row["n_estimators"],
    "max_depth": best_rf_row["max_depth"],
}

print("mejores params:", best_rf_params, "f1_macro val:", best_rf_row["f1_macro"])
best_rf_params


Reajustamos el bosque final solo con train y esos hiperparámetros.


In [ ]:
# train_features, train_labels, best_rf_params, seed

random_forest_model = RandomForestClassifier(
    n_estimators=best_rf_params["n_estimators"],
    max_depth=best_rf_params["max_depth"],
    class_weight="balanced",
    random_state=seed,
)
random_forest_model.fit(train_features, train_labels)

random_forest_model


Guardamos las métricas del bosque en val.


In [ ]:
# random_forest_model, val_features, val_labels

rf_val_predictions = random_forest_model.predict(val_features)
rf_val_f1 = f1_score(val_labels, rf_val_predictions, average="macro")
rf_val_report = classification_report(val_labels, rf_val_predictions)
rf_val_metrics = {
    "f1_macro": rf_val_f1,
    "report": rf_val_report,
}

print("f1_macro val (bosque):", rf_val_metrics["f1_macro"])
print(rf_val_metrics["report"])
rf_val_metrics


Comparamos en una tablita pequeña el f1 macro de val: mayoría, logística y bosque. Solo para leer quién ganó la validación.


In [ ]:
# majority_val_metrics, logistic_val_metrics, rf_val_metrics

val_comparison = pd.DataFrame(
    [
        {"modelo": "mayoria", "f1_macro_val": majority_val_metrics["f1_macro"]},
        {"modelo": "logistica", "f1_macro_val": logistic_val_metrics["f1_macro"]},
        {"modelo": "random_forest", "f1_macro_val": rf_val_metrics["f1_macro"]},
    ]
)

val_comparison


Ahora sí: escalamos test con el escalador de train (solo transform, sin reajustar).


In [ ]:
# feature_scaler, test_features

test_features_scaled = feature_scaler.transform(test_features)

print(test_features_scaled.shape)
test_features_scaled


Predicción única de la logística en test.


In [ ]:
# logistic_model, test_features_scaled

logistic_test_predictions = logistic_model.predict(test_features_scaled)

logistic_test_predictions


Reporte y matriz de confusión de la logística en test (una sola vez).


In [ ]:
logistic_test_f1 = f1_score(test_labels, logistic_test_predictions, average="macro")
logistic_test_report = classification_report(test_labels, logistic_test_predictions)
logistic_test_cm = confusion_matrix(test_labels, logistic_test_predictions)
logistic_test_metrics = {
    "f1_macro": logistic_test_f1,
    "report": logistic_test_report,
    "confusion_matrix": logistic_test_cm,
}

fig, ax = plt.subplots(figsize=(4, 3))
im = ax.imshow(logistic_test_cm, cmap="Blues")
ax.set_title("Logística HSV masked — confusión en test")
ax.set_xlabel("Predicho")
ax.set_ylabel("Real")
labels = sorted(test_labels.unique())
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_yticklabels(labels)
for i in range(logistic_test_cm.shape[0]):
    for j in range(logistic_test_cm.shape[1]):
        ax.text(j, i, int(logistic_test_cm[i, j]), ha="center", va="center")
fig.colorbar(im, ax=ax, fraction=0.046)
fig.tight_layout()
plt.show()

print("f1_macro test (logística):", logistic_test_metrics["f1_macro"])
print(logistic_test_metrics["report"])
print("matriz de confusión:\n", logistic_test_metrics["confusion_matrix"])
logistic_test_metrics


Predicción única del bosque en test.


In [ ]:
# random_forest_model, test_features

rf_test_predictions = random_forest_model.predict(test_features)

rf_test_predictions


Reporte y matriz de confusión del bosque en test (una sola vez).


In [ ]:
# test_labels, rf_test_predictions

rf_test_f1 = f1_score(test_labels, rf_test_predictions, average="macro")
rf_test_report = classification_report(test_labels, rf_test_predictions)
rf_test_cm = confusion_matrix(test_labels, rf_test_predictions)
rf_test_metrics = {
    "f1_macro": rf_test_f1,
    "report": rf_test_report,
    "confusion_matrix": rf_test_cm,
}

fig, ax = plt.subplots(figsize=(4, 3))
im = ax.imshow(rf_test_cm, cmap="Blues")
ax.set_title("Random Forest HSV masked — confusión en test")
ax.set_xlabel("Predicho")
ax.set_ylabel("Real")
labels = sorted(test_labels.unique())
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_yticklabels(labels)
for i in range(rf_test_cm.shape[0]):
    for j in range(rf_test_cm.shape[1]):
        ax.text(j, i, int(rf_test_cm[i, j]), ha="center", va="center")
fig.colorbar(im, ax=ax, fraction=0.046)
fig.tight_layout()
plt.show()

print("f1_macro test (bosque):", rf_test_metrics["f1_macro"])
print(rf_test_metrics["report"])
print("matriz de confusión:\n", rf_test_metrics["confusion_matrix"])
rf_test_metrics


Guardamos la comparación de val y las métricas de test en experiments/results con prefijo rocole_baseline_hsv_masked.


In [ ]:
# val_comparison, logistic_test_metrics, rf_test_metrics, best_logistic_C, best_rf_params

baseline_results_path = results_dir / "rocole_baseline_hsv_masked_metrics.csv"
val_comparison_path = results_dir / "rocole_baseline_hsv_masked_val_comparison.csv"

val_comparison.to_csv(val_comparison_path, index=False)

test_metrics_table = pd.DataFrame(
    [
        {
            "modelo": "logistica",
            "f1_macro_test": logistic_test_metrics["f1_macro"],
            "best_C": best_logistic_C,
        },
        {
            "modelo": "random_forest",
            "f1_macro_test": rf_test_metrics["f1_macro"],
            "n_estimators": best_rf_params["n_estimators"],
            "max_depth": best_rf_params["max_depth"],
        },
    ]
)
test_metrics_table.to_csv(baseline_results_path, index=False)

print("val:", val_comparison_path.resolve())
print("test metrics:", baseline_results_path.resolve())
baseline_results_path


Guardamos las figuras de confusión de test con el mismo prefijo hsv_masked.


In [ ]:
# logistic_test_metrics, rf_test_metrics, test_labels

logistic_cm_path = results_dir / "rocole_baseline_hsv_masked_logistic_confusion.png"
rf_cm_path = results_dir / "rocole_baseline_hsv_masked_rf_confusion.png"
labels = sorted(test_labels.unique())

fig, ax = plt.subplots(figsize=(4, 3))
im = ax.imshow(logistic_test_metrics["confusion_matrix"], cmap="Blues")
ax.set_title("Logística HSV masked — confusión en test")
ax.set_xlabel("Predicho")
ax.set_ylabel("Real")
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_yticklabels(labels)
cm = logistic_test_metrics["confusion_matrix"]
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, int(cm[i, j]), ha="center", va="center")
fig.colorbar(im, ax=ax, fraction=0.046)
fig.tight_layout()
fig.savefig(logistic_cm_path, dpi=150, bbox_inches="tight")
plt.close(fig)

fig, ax = plt.subplots(figsize=(4, 3))
im = ax.imshow(rf_test_metrics["confusion_matrix"], cmap="Blues")
ax.set_title("Random Forest HSV masked — confusión en test")
ax.set_xlabel("Predicho")
ax.set_ylabel("Real")
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_yticklabels(labels)
cm = rf_test_metrics["confusion_matrix"]
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, int(cm[i, j]), ha="center", va="center")
fig.colorbar(im, ax=ax, fraction=0.046)
fig.tight_layout()
fig.savefig(rf_cm_path, dpi=150, bbox_inches="tight")
plt.close(fig)

print("cm logística:", logistic_cm_path.resolve())
print("cm bosque:", rf_cm_path.resolve())
logistic_cm_path


## Conclusiones

Val f1_macro: mayoría 0.34 · logística 0.80 (C=0.1) · RF 0.83 (100 árboles).

Test, una sola vez, sobre las mismas 226 hojas: logística 0.68 · RF 0.74.

Comparado con el baseline masked solo RGB/GLCM (test ~0.61 log / ~0.68 RF), sumar los 16 bins HSV subió el f1_macro. No reabrimos ese test: la comparación es de memoria con las métricas ya guardadas.

Sigue sin ser un clasificador de campo listo, pero en desarrollo la familia HSV pequeña sí aportó sobre la máscara COCO. Antes de meter más familias, conviene revisar matrices de confusión y si en val se puede recortar bins sin perder el salto.
